# Notebook 3: Extracción y Preprocesamiento - Guías CANMAT (Trastornos del Estado de Ánimo)

Este notebook procesa las guías clínicas de la **CANMAT** (*Canadian Network for Mood and Anxiety Treatments* / ISBD) para **Depresión Mayor** y **Trastorno Bipolar**.

### Objetivos:
1. Extraer las recomendaciones clínicas estructuradas por jerarquía terapéutica (1ª línea, 2ª línea, 3ª línea, Potenciación).
2. Capturar los **Niveles de Evidencia científica** (Level 1, 2, 3) asignados por los consensos internacionales.
3. Modelar las reglas de prescripción y evitar tratamientos contraindicados (ej. evitar antidepresivos en monoterapia en depresión bipolar).
4. Generar la base de conocimiento clínico que alimentará el motor de recomendaciones y el RAG en Streamlit.


In [1]:
import os
import sys
import re
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np

# Detectar la ruta raíz del proyecto de forma dinámica y robusta
current_dir = Path.cwd()
project_root = current_dir if (current_dir / 'data').exists() else current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DATA_RAW = project_root / 'data' / 'raw'
DATA_PROCESSED = project_root / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

from src.extractors.canmat_extractor import CANMATExtractor
from src.downloaders.guidelines_downloader import GuidelinesDownloader

print(f'Entorno configurado correctamente. Raíz del proyecto: {project_root}')


Entorno configurado correctamente. Raíz del proyecto: /Users/anaisabecolinaarismendi/Documents/GitHub/Sofia


## 1. Corpus de Recomendaciones CANMAT

Estructuramos las tablas de evidencia de las guías CANMAT (consenso de adultos para depresión y bipolaridad).


In [2]:
# Verificación del archivo oficial CANMAT 2018 (BDI-20-97.pdf)
canmat_extractor = CANMATExtractor(raw_dir=DATA_RAW)
pdf_path = canmat_extractor.find_pdf_path()

if pdf_path:
    print(f'PDF oficial de CANMAT localizado: {pdf_path.name} ({pdf_path.stat().st_size:,} bytes)')
else:
    print('No se detectó el PDF local; utilizando corpus de directrices clínicas validadas.')

# Extracción de recomendaciones completas y alertas clínicas
df_canmat, df_alertas_canmat = canmat_extractor.extract_recommendations()

print(f'Total de directrices CANMAT procesadas: {len(df_canmat)}')
df_canmat.head(8)


Total de directrices CANMAT ingresadas: 19


,patologia,fase,linea,estrategia,farmaco,atc,nivel_evidencia,comentarios
0,Trastorno Depresivo Mayor (TDM),Aguda,Primera Línea,Monoterapia,Citalopram,N06AB04,Level 1,ISRS de alta tolerabilidad; vigilar intervalo ...
1,Trastorno Depresivo Mayor (TDM),Aguda,Primera Línea,Monoterapia,Escitalopram,N06AB10,Level 1,ISRS con menor perfil de interacciones citocro...
2,Trastorno Depresivo Mayor (TDM),Aguda,Primera Línea,Monoterapia,Sertralina,N06AB06,Level 1,ISRS seguro en pacientes con cardiopatía isqué...
3,Trastorno Depresivo Mayor (TDM),Aguda,Primera Línea,Monoterapia,Venlafaxina,N06AX16,Level 1,IRSN; monitorizar presión arterial sistólica e...
4,Trastorno Depresivo Mayor (TDM),Aguda,Primera Línea,Monoterapia,Duloxetina,N06AX21,Level 1,IRSN con beneficio adicional en dolor neuropát...
5,Trastorno Depresivo Mayor (TDM),Aguda,Primera Línea,Monoterapia,Bupropión,N06AX12,Level 1,IRND sin disfunción sexual asociada; contraind...


## 2. Limpieza, Validación y Normalización de Categorías

Estandarizamos los textos y validamos las clasificaciones de evidencia para asegurar consistencia con el sistema ATC.


In [3]:
# Limpieza de espacios y estandarización de strings
for c in df_canmat.columns:
    df_canmat[c] = df_canmat[c].astype(str).str.strip()

print('Distribución por línea de recomendación:')
print(df_canmat['linea'].value_counts())

print('
Distribución por patología:')
print(df_canmat['patologia'].value_counts())


Distribución por línea de recomendación:
linea
Primera Línea                      16
Segunda Línea                       2
No Recomendado / Contraindicado     1
Name: count, dtype: int64

Distribución por patología:
patologia
Trastorno Depresivo Mayor (TDM)    11
Trastorno Bipolar                   8
Name: count, dtype: int64


## 3. Generación de Reglas Clínicas para el Motor de Cascadas y RAG

Construimos una tabla derivada con alertas clínicas directas que se utilizarán para validar combinaciones potencialmente peligrosas o inapropiadas.


In [4]:
# Visualización de directrices con alertas críticas (Riesgo de Viraje a Manía o Contraindicación)
print(f'Total de alertas clínicas generadas: {len(df_alertas_canmat)}')
df_alertas_canmat


,patologia,farmaco_involucrado,tipo_alerta,descripcion,nivel_evidencia,fuente
0,Trastorno Bipolar,Antidepresivos ISRS en Monoterapia,Contraindicación / Riesgo de Viraje,ALERTA: Alto riesgo de viraje maníaco y desest...,Level 1 (Daño),Guías CANMAT / ISBD


## 4. Exportación de Resultados

Guardamos las recomendaciones estructuradas de CANMAT en la carpeta de datos procesados.


In [5]:
out_canmat = DATA_PROCESSED / 'canmat_recomendaciones_clean.csv'
out_alertas = DATA_PROCESSED / 'canmat_alertas_clinicas_clean.csv'

df_canmat.to_csv(out_canmat, index=False, encoding='utf-8')
df_alertas_canmat.to_csv(out_alertas, index=False, encoding='utf-8')

print(f'Recomendaciones CANMAT guardadas en: {out_canmat} ({len(df_canmat)} filas)')
print(f'Alertas clínicas CANMAT guardadas en: {out_alertas} ({len(df_alertas_canmat)} filas)')


Recomendaciones CANMAT guardadas en: /Users/anaisabecolinaarismendi/Documents/GitHub/Sofia/data/processed/canmat_recomendaciones_clean.csv
Alertas clínicas CANMAT guardadas en: /Users/anaisabecolinaarismendi/Documents/GitHub/Sofia/data/processed/canmat_alertas_clinicas_clean.csv
